# HW4 C: Text-to-speech
Implement a causal Transformer that predicts speech tokens from text. Frozen
CosyVoice2 audio models turn those tokens into speech. Use the full supplied
LibriTTS token cache and continue the provided starter for **one additional epoch**.

**43 points:** 16 for code, 24 for synthesized-speech WER, and 3 for report C1.
Code: dataset 2, collate 2, sequence construction 4, loss/causality 3, generation 2,
training 2, validation 1. Code credit is independent of model convergence.

| Synthesized-speech WER | Performance |
|---|---:|
| Below 15% | 24 |
| 15% to below 17% | 20 |
| 17% to below 19% | 16 |
| 19% to below 21% | 12 |
| 21% to below 22.5% | 8 |
| 22.5% to below 23.5% | 4 |
| 23.5% or higher | 0 |

The two-epoch starter measured 23.55% WER; the three-epoch staff reference reached
14.56%. Your results may differ. Missing predictions count as deletions.

**Report:** use the [template](https://github.com/Louis0324/eecs183-hw4/blob/main/report_template.txt)
for one selectable-text `report.pdf` covering **A1, A2, A3, B1, C1**. Answer the
prompts in 3–5 sentences each, plus requested figures/sequences. Write
`Not completed` for unfinished work; put conceptual answers in the PDF.


## Setup
Save a copy in Drive, select a Colab GPU, and run cells in order. Setup and data
downloads are automatic; no Hugging Face login is needed. If pip requests a
restart, restart and rerun setup.

Implement the TODOs using their docstrings. Keep the specified top-level
function/class names and signatures. Put extra imports and constants inside
functions/classes: the grader does not execute other top-level code.

Check your implementation with `SMOKE_TEST=True`, then set it to `False` for
experiments. Save notebooks, results, and checkpoints to Drive or download them;
Colab local storage is temporary.

| Implement | Provided |
|---|---|
| Dataset and collator | Cache loading and data loaders |
| Sequence construction, logits, loss, generation | Exact model constructor and starter loader |
| `train_epoch` and `validate` | Optimizer, scheduler, epoch loop, checkpoints/resume |
| Answer C1 | Audio synthesis, Whisper evaluation, submission export |

Keep the supplied constructor unchanged for checkpoint compatibility. In smoke
mode, check finite metrics, changing parameters during training, and unchanged
parameters during validation. Then use normal mode, train, and evaluate all
200 test texts with your voice prompt.


In [ ]:
REQUIREMENTS = 'requirements-tts.txt'
# Colab: select Runtime > Change runtime type > GPU before training.
import sys, subprocess
from pathlib import Path

# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v3'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v3', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQUIREMENTS])
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_TEST = False  # True: only a few steps, for debugging your implementation.
DATA_ROOT = Path(os.environ.get('HW4_DATA_ROOT', 'data'))
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
# Checkpoints/results disappear when Colab deletes a runtime. Copy them to Drive
# or download them before disconnecting. Submission ZIPs contain no checkpoints.

In [ ]:
# Download only Part C data; verified cached copies are reused.
from course_download import ensure_course_data
DATA_ARCHIVE = None  # Optional offline override: Path("hw4-data-c.zip")
ensure_course_data('c', data_root=DATA_ROOT, archive=DATA_ARCHIVE)


In [ ]:
import json, math, copy
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, unpad_sequence
from sklearn.metrics import accuracy_score, r2_score
import matplotlib.pyplot as plt
from hw4_runtime import load_phoneme_data, load_mfcc_data, load_audio
np.random.seed(183); torch.manual_seed(183)
device = DEVICE

from IPython.display import Audio, display
class Config:
    IGNORE_ID = -1
    SOS_EOS_ID = 0  # Special-embedding index; NOT speech EOS.
    TASK_ID = 1    # Special-embedding index; NOT a speech-token offset.
    DATA_CACHE_DIR = str(DATA_ROOT / 'tts')
    PRETRAINED_DIR = os.environ.get('HW4_PRETRAINED_DIR', 'pretrained_models')
    RESULTS_DIR = str(RESULTS)

## 1. Load data and the text tokenizer
Training uses cached speech tokens. Audio models load later for synthesis to save
GPU memory. Use the text tokenizer's **full vocabulary**, including added tokens.


In [ ]:
from hw4_util import download_pretrained_models, load_text_tokenizer
model_dir = download_pretrained_models(Config.PRETRAINED_DIR, tokenizer_only=True)
text_tokenizer = load_text_tokenizer(model_dir)
train_cache = torch.load(DATA_ROOT / 'tts/libritts_train.pt', map_location='cpu', weights_only=True)
val_cache = torch.load(DATA_ROOT / 'tts/libritts_dev.pt', map_location='cpu', weights_only=True)
train_data, val_data = train_cache['samples'], val_cache['samples']
speaker_to_idx = train_cache['metadata']['speaker_to_idx']
speech_vocab_size = train_cache['metadata']['speech_vocab_size']
print('Train:', len(train_data), 'Validation:', len(val_data), 'Speech vocabulary:', speech_vocab_size)

## 2. Dataset and batching
Samples contain `utt_id`, `text`, `speaker_id`, and `speech_tokens`.

- Return long token tensors; reject empty/invalid samples or samples exceeding
  either limit (512 text tokens, 1,200 speech tokens) with `None`.
- Preserve text/audio pairing; do not truncate them independently. Unknown
  validation speakers use `speaker_idx=-1` (metadata only).
- The collator filters `None`, sorts all fields by decreasing speech length, and
  pads tokens with zero. Return `None` if all samples are invalid.

Use the full training cache outside smoke mode. Follow the return fields in the docstrings.


In [ ]:
class CosyVoiceDataset(Dataset):

    def __init__(self, samples_list, text_tokenizer, speaker_to_idx_dict, max_text_len=512, max_speech_len=1200):
        self.samples = samples_list
        self.text_tokenizer = text_tokenizer
        self.speaker_to_idx = speaker_to_idx_dict
        self.max_text_len = max_text_len
        self.max_speech_len = max_speech_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """Return dict: utt, text, text_token (long [T]), speech_token (long [S]), speaker_idx, speaker_id. Follow the validation and maximum-length rules above. Preserve the audio/text pairing."""
        raise NotImplementedError('TODO: __getitem__')

In [ ]:
def cosyvoice_collate_fn(batch):
    """Return None for an all-invalid batch; otherwise sort by speech length descending and return utts, text, text_tokens, text_lengths, speech_tokens, speech_lengths, speaker_indices. Pad token tensors with 0; return lengths and indices as int64. Reorder every field consistently."""
    raise NotImplementedError('TODO: cosyvoice_collate_fn')

In [ ]:
train_dataset = CosyVoiceDataset(train_data[:16] if SMOKE_TEST else train_data, text_tokenizer, speaker_to_idx)
val_dataset = CosyVoiceDataset(val_data[:8] if SMOKE_TEST else val_data, text_tokenizer, speaker_to_idx)
train_loader = DataLoader(train_dataset, batch_size=2 if SMOKE_TEST else 4, shuffle=True, num_workers=0, collate_fn=cosyvoice_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2 if SMOKE_TEST else 4, shuffle=False, num_workers=0, collate_fn=cosyvoice_collate_fn)
first_batch = next(x for x in train_loader if x is not None)
print({k: tuple(v.shape) for k, v in first_batch.items() if torch.is_tensor(v)})

## 3. Causal speech-token LM
Use separate text, speech, and special-token embeddings. Speech IDs are
`0..speech_vocab_size-1`; speech **EOS = speech_vocab_size**. SOS is index 0 in
the separate special-token table.

For text `[t0,t1]` and speech `[s0,s1,s2]`:

```text
Inputs:  [SOS,t0,t1,TASK,s0,s1,s2]
Targets: [-1, -1, -1, s0,s1,s2,EOS]
```

TASK predicts the first speech token; the last speech input predicts EOS. Ignore
prefix/padded targets (`IGNORE_ID=-1`). Use causal and padding masks (`True`
means blocked); compute loss/accuracy only on non-ignored targets.
`_logits` adds positions and returns raw `[B,T,speech_vocab_size+1]` logits.

Keep the provided constructor/module names: **768 dimensions, 12 heads,
12 layers, 2,048 positions**. Honor dimension arguments for the grader's small
CPU checks. Smoke mode reduces data/steps, not the architecture.


In [ ]:
class TextToSpeechLM(nn.Module):

    def __init__(self, text_vocab_size, speech_vocab_size, d_model=768, n_heads=12, n_layers=12, max_seq_len=2048):
        super().__init__()
        self.model_config = dict(d_model=d_model, n_heads=n_heads, n_layers=n_layers, max_seq_len=max_seq_len)
        self.d_model = d_model
        self.speech_vocab_size = speech_vocab_size
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        self.speech_embedding = nn.Embedding(speech_vocab_size + 1, d_model)
        self.special_embedding = nn.Embedding(2, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, dropout=0.0, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.output_projection = nn.Linear(d_model, speech_vocab_size + 1)

    def prepare_sequence(self, text_tokens, text_lengths, speech_tokens=None, speech_lengths=None):
        """Inputs: padded text/speech IDs and valid lengths. Return input embeddings [B,L,D], int64 targets [B,L], bool padding mask [B,L]. Follow the exact shift described above. If speech is None, return the text prefix and all ignored targets. Preserve embedding gradients."""
        raise NotImplementedError('TODO: prepare_sequence')

    def _logits(self, inputs, padding=None):
        """Add learned position embeddings. Apply a boolean upper-triangular causal mask and optional boolean padding mask to the Transformer. Return raw logits [B,L,V+1]. Reject sequences longer than max_seq_len."""
        raise NotImplementedError('TODO: _logits')

    def forward(self, text_tokens, text_lengths, speech_tokens, speech_lengths):
        """Call prepare_sequence and _logits; compute scalar cross entropy with ignore_index=-1 and scalar accuracy on non-ignored targets. Return (loss, accuracy)."""
        raise NotImplementedError('TODO: forward')

    @torch.no_grad()
    def generate(self, text_tokens, max_length=500, temperature=1.0, top_k=50, prompt_speech_token=None, prompt_text_tokens=None, min_length=None):
        """Batch size 1. Set eval mode and disable gradients. Prefix: SOS, prompt text (if any), target text, TASK, prompt speech (if any). Repeatedly call _logits, sample using temperature>0 and top-k (clamp k to vocabulary size; 0 disables top-k). Block EOS before min_length; stop at EOS or max_length new tokens. Return long [1,N] containing ONLY new speech tokens, excluding prompt and EOS. EOS ID is speech_vocab_size."""
        raise NotImplementedError('TODO: generate')


In [ ]:
# Keep the original architecture in smoke mode; only reduce the data/steps.
MODEL_KWARGS = dict(d_model=768, n_heads=12, n_layers=12, max_seq_len=2048)
model = TextToSpeechLM(text_tokenizer.vocab_size, speech_vocab_size, **MODEL_KWARGS).to(device)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))
# Use the course starter for normal training; smoke mode needs no checkpoint download.
# For your own interrupted training, set RESUME=True and keep the same RESULTS directory.
RESUME = False
USE_STARTER = not SMOKE_TEST
STARTER_CHECKPOINT = os.environ.get('HW4_TTS_STARTER')  # Optional local file override.
starter_metadata = None
if USE_STARTER and not RESUME:
    from tts_checkpoint import load_tts_starter
    from course_download import download_course_file
    STARTER_CHECKPOINT = download_course_file("hw4-tts-starter.pt", local_path=STARTER_CHECKPOINT)
    starter_metadata = load_tts_starter(model, text_tokenizer, STARTER_CHECKPOINT)
    print('Loaded course starter; completed reference epochs:',
          starter_metadata['source_training']['completed_epochs'])


## 4. Training and validation
Implement the functions below:

- **Training:** skip `None` batches, move tensors to the device, use train mode,
  clear gradients, backpropagate, clip gradients, and step the optimizer and
  scheduler once per valid batch (`scheduler` may be `None`).
- **Validation:** use eval mode and no gradients; leave parameters unchanged.
- Return mean loss/accuracy over valid batches, or `(0.0, 0.0)` if none are valid.

The optimizer, scheduler, epoch loop, and checkpoint selection are provided.
Defaults: AdamW, learning rate `2e-4`, weight decay `0.01`, 10% warmup then cosine
decay. Tuning is optional; use validation results. Reduce batch size if needed.

**Warm start:** the model cell automatically loads the two-epoch starter. Use a
fresh optimizer and train one additional epoch; `num_epochs` counts student
epochs only. Smoke mode skips the starter. `USE_STARTER=False` trains from scratch.
For your own saved training state, use `RESUME=True` with the same architecture
and settings. Use a new results directory for a new experiment. If compute ends
early, save your work and report the completed training.


In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    """Implement one epoch with the exact behavior above. Move text_tokens, text_lengths, speech_tokens, speech_lengths to device. scheduler may be None. Return (mean_loss, mean_accuracy)."""
    raise NotImplementedError('TODO: train_epoch')

In [ ]:
def validate(model, dataloader, device):
    """Evaluate valid batches with model.eval() and torch.no_grad(). No updates or gradients. Return (mean_loss, mean_accuracy), or (0.0,0.0) for no valid batches."""
    raise NotImplementedError('TODO: validate')

In [ ]:
from hw4_util import get_warmup_cosine_scheduler, save_checkpoint
if USE_STARTER and not RESUME and starter_metadata is None:
    raise RuntimeError('Load the course starter successfully before training, or set USE_STARTER=False.')
num_epochs = 1 if (SMOKE_TEST or USE_STARTER) else 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
total_steps = num_epochs * len(train_loader)
scheduler = get_warmup_cosine_scheduler(optimizer, max(1, total_steps // 10), total_steps)
best_val_loss, start_epoch = float('inf'), 0
if RESUME:
    if not (RESULTS / 'tts_latest.pt').exists():
        raise FileNotFoundError('No tts_latest.pt in RESULTS; restore your saved checkpoint or set RESUME=False.')
    state = torch.load(RESULTS / 'tts_latest.pt', map_location='cpu', weights_only=True)
    if state.get('model_config') != MODEL_KWARGS:
        raise ValueError('Resume checkpoint architecture does not match the original TTS model.')
    model.load_state_dict(state['model_state_dict'], strict=True)
    starter_metadata = state.get('starter_metadata')
    optimizer.load_state_dict(state['optimizer_state_dict'])
    if state.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(state['scheduler_state_dict'])
    start_epoch = state['epoch'] + 1
    if (RESULTS / 'tts_best.pt').exists():
        best_val_loss = torch.load(RESULTS / 'tts_best.pt', map_location='cpu', weights_only=True)['val_loss']
for epoch in range(start_epoch, num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc = validate(model, val_loader, device)
    print(f'Epoch {epoch+1}: train={train_loss:.4f}, dev={val_loss:.4f}, dev accuracy={val_acc:.2%}')
    save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, RESULTS / 'tts_latest.pt', model_config=MODEL_KWARGS, starter_metadata=starter_metadata)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, None, None, epoch, train_loss, val_loss, RESULTS / 'tts_best.pt', model_config=MODEL_KWARGS, starter_metadata=starter_metadata)


## 5. Check audio decoding
First decode ground-truth tokens to check the audio pipeline before testing your
LM. This cell downloads frozen audio models and frees training optimizer state
to save memory. Recreate the optimizer by rerunning the training setup if you
return to training later.


In [ ]:
# Checkpoints are saved. Release training state before loading the audio models.
import gc
model.zero_grad(set_to_none=True)
globals().pop('scheduler', None)
globals().pop('optimizer', None)
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

from hw4_util import load_speech_tokenizer, load_flow_model, load_vocoder, generate_audio_from_tokens, synthesize
model_dir = download_pretrained_models(Config.PRETRAINED_DIR)
speech_tokenizer = load_speech_tokenizer(model_dir, device)
flow_model = load_flow_model(model_dir, device)
vocoder = load_vocoder(model_dir, device)
sample = next(x for x in (val_dataset[i] for i in range(len(val_dataset))) if x is not None)
audio, mel = generate_audio_from_tokens(sample['speech_token'][:50] if SMOKE_TEST else sample['speech_token'], flow_model, vocoder, device)
display(Audio(audio.cpu().numpy(), rate=24000))
state = torch.load(RESULTS / 'tts_best.pt', map_location='cpu', weights_only=True)
model.load_state_dict(state['model_state_dict']); model.eval()
audio, sr = synthesize('This is a short speech synthesis test.', model, text_tokenizer, speech_tokenizer, flow_model, vocoder, device,
                       output_path=RESULTS / 'tts_example.wav', max_length=20 if SMOKE_TEST else 500)
display(Audio(audio.numpy(), rate=sr))

## 6. Voice prompt and evaluation
Use the best validation checkpoint. Upload **3–10 seconds of your speech** as
WAV/FLAC with its **exact transcript** (or set `audio_path`). The helper resamples
to mono 16 kHz.

Synthesize all **200 test texts**, transcribe them with Whisper, and save
`RESULTS/tts_submission.json` with the original text IDs. Submit Whisper's outputs
unchanged. Audio stays local for listening and is excluded from the ZIP.

Rerun to resume. New model weights or prompts start fresh predictions. After
changing decoding code/settings, use `RESTART_PREDICTIONS=True` once, then `False`
to resume interrupted evaluation.


In [ ]:
from hw4_util import record_or_load_voice, run_voice_cloning_evaluation
prompt_audio, prompt_text, recording_path = record_or_load_voice(
    prompt_text='The quick brown fox jumps over the lazy dog.', results_dir=RESULTS)
STUDENT_ID = 'YOUR_STUDENT_ID'  # Replace this; no ID-based shuffling is performed.
RESTART_PREDICTIONS = False  # True after changing generation code/settings.
submission_path = run_voice_cloning_evaluation(
    STUDENT_ID, model, text_tokenizer, speech_tokenizer, flow_model, vocoder,
    prompt_audio, prompt_text, results_dir=RESULTS, pretrained_dir=Config.PRETRAINED_DIR,
    device=device, model_dir=model_dir, max_examples=2 if SMOKE_TEST else None, restart=RESTART_PREDICTIONS)
print('Saved', submission_path)

## Report C1: Teacher forcing, masks, and WER (3 points)
Answer in **C1** of `report.pdf` (3–5 sentences plus sequences; no extra training):

1. For text `[t0]` and speech `[s0,s1]`, write the input and target sequences,
   including SOS, TASK, EOS, and ignored targets. Which position predicts the
   first speech token? (1 point)
2. What leaks without a causal mask, and why exclude padded positions? (1 point)
3. Why does low synthesized-speech WER not establish similarity to the prompt
   speaker's voice? (1 point)


## Prepare your submission
Submit **one combined ZIP with one shared `report.pdf`**. The final cell includes
your current notebook and available predictions/plots; it excludes checkpoints,
datasets, and audio. Combine separate reports into one PDF yourself.

1. Leave `include_previous=True`. Upload `hw4-ab.zip` at the earlier-ZIP prompt
   (or select separate A and B ZIPs together). For C only, cancel this prompt.
2. Upload your final combined `report.pdf`, or cancel to keep an earlier PDF.
3. Upload the downloaded **`submission.zip`** to Gradescope.

**Partial work is accepted:** completed questions are graded independently.
Check the printed file list and your PDF before uploading. Every Gradescope
resubmission must contain **all** work you want graded.

**Another order or runtime reset:** set `include_previous=True` and import your
latest ZIP in whichever notebook you package last. The final cell runs without
training/setup or a GPU. For results saved on Drive, mount Drive and restore
`RESULTS` first. Use the latest version of each part; keep the downloaded ZIP.

Saved predictions: `RESULTS/tts_submission.json` (200 test texts).


In [ ]:
include_report = True  # Upload one combined PDF, or cancel to retain an earlier one.
include_previous = True  # Merge earlier submission ZIPs; cancel if none.

# Packaging needs no training dependencies, GPU, or data download.
# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v3'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v3', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION, packager_only=True)
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
from make_submission import build_submission, from_colab
if colab_files is not None:
    from_colab('c', include_report=include_report, include_previous=include_previous,
               results_dir=globals().get("RESULTS", "results"))
else:
    build_submission(report="report.pdf" if include_report else None)
